# SDRF Extraction Pipeline Kaggle: Harmonizing the Data of Your Data

rules + LLM + normalisation (tbd iterative)
plain langchain, no dspy

In [1]:
import time
from datetime import timedelta
start_time = time.monotonic()

# Dependencies, configuration, LocalAI

In [2]:
# ── Kaggle secrets ───────────────────────────────────────────────────────────
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    def get_secret_safe(name, default=None):
        try: return user_secrets.get_secret(name)
        except Exception: return default
    api_key  = get_secret_safe('LLM_API_KEY', 'sk-dummy')
    api_base = get_secret_safe('LLM_API_BASE', None)
    LLM_MODEL = get_secret_safe("LLM_MODEL", None)
    # Force into environment (CRITICAL)
    os.environ["OPENAI_API_KEY"] = api_key
    # (Optional but helps LiteLLM routing)
    os.environ["OPENAI_API_BASE"] = "https://api.openai.com/v1"
except Exception as x:
    print(x)
    api_key = None
    api_base = None
    LLM_MODEL = None
if LLM_MODEL is None:
    api_key = 'sk-dummy'
    api_base = "http://127.0.0.1:8080/v1"
    CONTEXT_SIZE = 24576
    # Tested models:
    # gemma-3-4b-it @ kaggle (GPU P100)
    # medgemma-4b-it @ kaggle (GPU P100)
    # deepseek-coder-v2-lite-instruct @ kaggle (GPU T4 x 2)
    LLM_MODEL = "gemma-3-4b-it"

print(f'api_base={api_base}, api_key={api_key[:4]}, model={LLM_MODEL}')

api_base=http://127.0.0.1:8080/v1, api_key=sk-d, model=gemma-3-4b-it


In [3]:
LAUNCH_LOCAL_AI = api_base is None or api_base == "http://127.0.0.1:8080/v1"
LAUNCH_LOCAL_AI

True

## LocalAI Server

In [4]:
# Create tmp folder to be used as a larger scratch space
from pathlib import Path

TMP_DIR = Path('../tmp')
TMP_DIR.mkdir(exist_ok=True)

In [5]:
%%bash
# Download and make executable the local-ai binary (https://github.com/mudler/LocalAI/releases)
cd ../tmp
if [ ! -f ./local-ai-* ]; then
    wget -q https://github.com/mudler/LocalAI/releases/download/v4.0.0/local-ai-v4.0.0-linux-amd64
fi
chmod +x local-ai-*

In [6]:
%%bash
# Download Model Gallery
mkdir /kaggle/tmp/models
cd /kaggle/tmp/models
git clone --filter=blob:none --sparse https://github.com/mudler/LocalAI.git
cd LocalAI
git sparse-checkout set gallery

Cloning into 'LocalAI'...


In [7]:
import subprocess
import requests
import time
import signal
import os

# Use modified local model gallery
os.environ["GALLERIES"] = '[{"name":"localai", "url":"file:///kaggle/tmp/models/LocalAI/gallery/index.updated.yaml"}]'

class LocalAIServer:
    def __init__(self, binary_path: str, model_name: str, host: str = "127.0.0.1", port: int = 8080):
        self.binary_path = binary_path
        self.model_name = model_name
        self.host = host
        self.port = port
        self.proc = None

    def start(self):
        """Start LocalAI as a subprocess."""
        if self.proc is not None:
            raise RuntimeError("Server already running")
        
        self.proc = subprocess.Popen(
            [self.binary_path, "run", self.model_name, "--disable-web-ui"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        self._wait_until_ready()

    def _wait_until_ready(self, timeout: int = 1200):
        """Poll the server until it responds or timeout."""
        url = f"http://{self.host}:{self.port}/v1/models"
        start_time = time.time()
        while True:
            try:
                res = requests.get(url, timeout=1)
                if res.status_code == 200:
                    print("LocalAI server is ready!")
                    break
            except requests.RequestException:
                pass  # server not ready yet
            if time.time() - start_time > timeout:
                raise TimeoutError("LocalAI server did not start in time")
            time.sleep(1)

    def stop(self):
        """Stop the LocalAI server."""
        if self.proc:
            self.proc.terminate()  # SIGTERM
            try:
                self.proc.wait(timeout=5)
            except subprocess.TimeoutExpired:
                self.proc.kill()   # SIGKILL if needed
                self.proc.wait()
            self.proc = None
            print("LocalAI server stopped.")

In [8]:
# Set the context_size for the selected model from the local model gallery

import yaml

def set_context_size(yaml_path, model_name, context_size, output_path=None):
    with open(yaml_path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    if not isinstance(data, list):
        raise ValueError("Expected a list of models")

    found = False

    for model in data:
        if model.get("name") == model_name:
            # Ensure overrides block exists
            overrides = model.get("overrides")
            if overrides is None:
                overrides = {}
                model["overrides"] = overrides

            # Set / update context_size
            overrides["context_size"] = context_size

            found = True
            break

    if not found:
        raise ValueError(f"Model '{model_name}' not found")

    out = output_path or yaml_path
    with open(out, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False)

    return out

set_context_size(
    yaml_path = "/kaggle/tmp/models/LocalAI/gallery/index.yaml",
    model_name = LLM_MODEL,
    context_size = CONTEXT_SIZE,
    output_path = "/kaggle/tmp/models/LocalAI/gallery/index.updated.yaml"
)

'/kaggle/tmp/models/LocalAI/gallery/index.updated.yaml'

In [9]:
if LAUNCH_LOCAL_AI:
    server = LocalAIServer("/kaggle/tmp/local-ai-v4.0.0-linux-amd64", LLM_MODEL)
    os.chdir( "/kaggle/tmp")
    server.start()

    """"
    # Now the server is ready, you can query it:
    res = requests.post(
        "http://127.0.0.1:8080/v1/chat/completions",
        json={
            "model": LLM_MODEL,
            "messages": [{"role": "user", "content": "Explain LLMs in one sentence."}]
        },
        timeout=1200
    )
    print(res.json())
    """
else:
    """
    res = requests.post(
        f"{api_base}/chat/completions",
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json"
        },        
        json={
            "model": LLM_MODEL,
            "messages": [{"role": "user", "content": "Explain LLMs in one sentence."}]
        },
        timeout=1200
    )    
    print(res.status_code)
    print(res.text)
    """
    server = None

LocalAI server is ready!


In [10]:
import logging

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('sdrf')

In [11]:
print("CWD:", os.getcwd())
os.chdir("/kaggle/working")
!pwd

CWD: /kaggle/tmp
/kaggle/working


##  Install & Import modelmess

In [12]:
%pip install -q langchain langchain-openai langchain-core pydantic openai sdrf-pipelines[all] sdrf-pipelines[ontology] PyYAML 
# check if ols dep is in
from sdrf_pipelines.ols.ols import OlsClient, OLS_AVAILABLE
print(OLS_AVAILABLE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 569.0/569.0 kB 38.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


2026-03-30 11:12:34,609 INFO NumExpr defaulting to 4 threads.


True


In [13]:
import sys

os.chdir("/kaggle/working")
!cd /kaggle/working

repo_url = 'https://github.com/vedina/modelmess.git'
repo_dir = '/kaggle/working/modelmess'

#branch = "iterative"
branch = "main"

if os.path.exists(repo_dir):
    print("Repo already exists. Pulling latest changes...")
    !cd {repo_dir} && git pull
else:
    print(f"Cloning repo (branch: {branch})...")
    !git clone --branch {branch} --single-branch {repo_url}

# Add the src folder inside the repo to Python path
SOURCE_DIR = f"{repo_dir}/sdrf_pipeline"
# Remove any old modelmess paths (important)
sys.path = [p for p in sys.path if "modelmess" not in p]
# Add correct one at highest priority
sys.path.insert(0,SOURCE_DIR)

# Verify import
from src.rules_0000 import PaperJSON

print('modelmess sdrf_pipeline imported OK')
%cd {SOURCE_DIR}
!pwd

Repo already exists. Pulling latest changes...
Already up to date.
modelmess sdrf_pipeline imported OK
/kaggle/working/modelmess/sdrf_pipeline
/kaggle/working/modelmess/sdrf_pipeline


## Configuration

In [14]:
ON_KAGGLE = Path('/kaggle').exists()

# ── LLM config ───────────────────────────────────────────────────────────────
LLM_API_BASE = api_base           # from Cell 0
LLM_API_KEY  = api_key
LLM_MODEL    = LLM_MODEL
RUN = 1

LLM_BACKEND = 'openai_compat'

TEMPERATURE = 0

BASE_DIR       = Path('/kaggle/input/competitions/harmonizing-the-data-of-your-data')
TRAIN_TEXT_DIR = BASE_DIR / 'Training_PubText' / 'PubText'
TRAIN_SDRF_DIR = BASE_DIR / 'Training_SDRFs' / 'HarmonizedFiles'
TEST_TEXT_DIR  = BASE_DIR / 'Test PubText' / 'Test PubText'
SAMPLE_SUB     = BASE_DIR / 'SampleSubmission.csv'
OUTPUT_DIR     = Path(f'/kaggle/working/output_sdrfs/{LLM_MODEL}/T{TEMPERATURE}/R{RUN}')
OUTPUT_DIR_RULES = Path(f'/kaggle/working/output_sdrfs/rules_0000')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_RULES.mkdir(parents=True, exist_ok=True)

SUBMISSION = "/kaggle/working/submission.csv"

print(f'Backend : {LLM_BACKEND}')
print(f'URL     : {LLM_API_BASE}')
print(f'Model   : {LLM_MODEL}')
print(f'Kaggle  : {ON_KAGGLE}')
print(f'Output  : {OUTPUT_DIR}')

Backend : openai_compat
URL     : http://127.0.0.1:8080/v1
Model   : gemma-3-4b-it
Kaggle  : True
Output  : /kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1


# Run extraction pipeline


In [15]:
!python main_fill.py --help

usage: main_fill.py [-h] [--pattern GLOB] [--stage {rules,llm,both}]
                    [--rules-only] [--rules-dir RULES_DIR] [--llm-dir LLM_DIR]
                    [--fill-from DIR] [--api-key API_KEY]
                    [--base-url BASE_URL] [--model MODEL]
                    [--max-tokens MAX_TOKENS] [--context-limit TOKENS]
                    [--no-dedup] [--prompts TOML] [--dump-prompts TOML]
                    [--verbose]
                    [input]

SDRF extraction -- rules + LLM gap-fill pipeline.

positional arguments:
  input                 Paper JSON source. Two forms: directory/ -> all *.json
                        files in that folder path/to/file.json -> single file
                        Use --pattern to filter files inside a directory. Not
                        required when using --dump-prompts.

options:
  -h, --help            show this help message and exit
  --pattern GLOB        Glob pattern inside the input directory (default:
                        

## Bootstrap with rules 
adapted from https://www.kaggle.com/code/nikitagajbhiye30/harmonizing-0000

In [16]:
log.info(f"Writing into {OUTPUT_DIR}")
log.info(f"Reading from {TEST_TEXT_DIR}")
pxd_files_only = f"{TEST_TEXT_DIR}/PXD*.json"

#{TEST_TEXT_DIR}/PXD*.json
!python main_fill.py "{TEST_TEXT_DIR}" --pattern PXD*.json  --stage rules --rules-dir {OUTPUT_DIR_RULES}

2026-03-30 11:12:39,786 INFO Writing into /kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1
2026-03-30 11:12:39,788 INFO Reading from /kaggle/input/competitions/harmonizing-the-data-of-your-data/Test PubText/Test PubText


11:12:39 [INFO] Pattern 'PXD*.json' in /kaggle/input/competitions/harmonizing-the-data-of-your-data/Test PubText/Test PubText -> 15 file(s)
11:12:39 [INFO] -- [1/15] PXD004010_PubText.json --
11:12:40 [INFO] [rules] PXD004010_PubText.json -> 10 rows, 54/73 fields filled in row 0
11:12:54 [INFO] Written 10 rows -> /kaggle/working/output_sdrfs/rules_0000/PXD004010_PubText.sdrf.csv
11:12:54 [INFO] -- [2/15] PXD016436_PubText.json --
11:12:54 [INFO] [rules] PXD016436_PubText.json -> 18 rows, 62/73 fields filled in row 0
11:12:54 [INFO] Written 18 rows -> /kaggle/working/output_sdrfs/rules_0000/PXD016436_PubText.sdrf.csv
11:12:54 [INFO] -- [3/15] PXD019519_PubText.json --
11:12:54 [INFO] [rules] PXD019519_PubText.json -> 6 rows, 64/73 fields filled in row 0
11:12:54 [INFO] Written 6 rows -> /kaggle/working/output_sdrfs/rules_0000/PXD019519_PubText.sdrf.csv
11:12:54 [INFO] -- [4/15] PXD025663_PubText.json --
11:12:54 [INFO] [rules] PXD025663_PubText.json -> 12 rows, 60/73 fields filled in ro

In [17]:
MAX_TOKENS = 8192
#CONTEXT_LIMIT = CONTEXT_SIZE
CONTEXT_LIMIT = 22912
log.info(f"Writing into {OUTPUT_DIR}")
log.info(f"Reading from {TEST_TEXT_DIR}")
log.info(f"{api_base} {LLM_MODEL}")

2026-03-30 11:12:57,369 INFO Writing into /kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1
2026-03-30 11:12:57,370 INFO Reading from /kaggle/input/competitions/harmonizing-the-data-of-your-data/Test PubText/Test PubText
2026-03-30 11:12:57,371 INFO http://127.0.0.1:8080/v1 gemma-3-4b-it


## Running LLM stage (gap filling)

> python main_fill.py papers/ --stage llm  --fill-from output/rules --llm-dir output/llm_gpt4o  --model gpt-4o --api-key $OPENAI_API_KEY

In [18]:
# Running second (LLM) stage (gap filling)
!python main_fill.py --stage llm --fill-from {OUTPUT_DIR_RULES} --max-tokens {MAX_TOKENS} --context-limit {CONTEXT_LIMIT} --llm-dir {OUTPUT_DIR}  --model {LLM_MODEL} --base-url {api_base} --api-key {api_key} --pattern PXD*.json "{TEST_TEXT_DIR}"

11:12:57 [INFO] Pattern 'PXD*.json' in /kaggle/input/competitions/harmonizing-the-data-of-your-data/Test PubText/Test PubText -> 15 file(s)
11:12:57 [INFO] -- [1/15] PXD004010_PubText.json --
11:12:57 [INFO] [llm] Loading rules CSV from /kaggle/working/output_sdrfs/rules_0000/PXD004010_PubText.sdrf.csv
11:13:04 [INFO] Deduplicated: 10 row(s) → 1 unique N/A pattern(s)
11:13:04 [INFO] Filling 62 field(s) for group of 10 row(s)…
11:13:04 [INFO] Chars: Paper9630 = M8027+A1493+T110
11:13:04 [INFO] Tokens: Paper2406 = M2006+A373+T27
11:13:04 [INFO] Context limit 22912 Available 10498 = CT22912-MT8192-FT4222
11:13:57 [INFO] HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.1 200 OK"
11:14:17 [INFO] HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.1 200 OK"
11:14:17 [INFO] Response received len=1848
11:14:17 [INFO] [llm]   PXD004010_PubText.json -> 45/73 fields filled in row 0 (+25 via LLM)
11:14:17 [INFO] Written 10 rows -> /kaggle/working/output_sdrfs/g

## Next iteration(s) of LLM stage (gap filling)

In [19]:
# Preparing to run next iteration of second (LLM) stage (gap filling)
RUN=1
print((RUN, OUTPUT_DIR))

DIR_RUN2 = str(OUTPUT_DIR).replace(f"R{RUN}",f"R{(RUN+1)}")
Path(DIR_RUN2).mkdir(parents=True, exist_ok=True)
RUN+1, DIR_RUN2

(1, PosixPath('/kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1'))


(2, '/kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R2')

In [20]:
# Running next iteration of second (LLM) stage (gap filling)

!python main_fill.py --stage llm --fill-from {OUTPUT_DIR} --max-tokens {MAX_TOKENS} --context-limit {CONTEXT_LIMIT} --llm-dir "{DIR_RUN2}"  --model {LLM_MODEL} --base-url {api_base} --api-key {api_key} --pattern PXD*.json "{TEST_TEXT_DIR}"

11:27:08 [INFO] Pattern 'PXD*.json' in /kaggle/input/competitions/harmonizing-the-data-of-your-data/Test PubText/Test PubText -> 15 file(s)
11:27:08 [INFO] -- [1/15] PXD004010_PubText.json --
11:27:08 [INFO] [llm] Loading rules CSV from /kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD004010_PubText.sdrf.csv
11:27:15 [INFO] Deduplicated: 10 row(s) → 1 unique N/A pattern(s)
11:27:15 [INFO] Filling 37 field(s) for group of 10 row(s)…
11:27:15 [INFO] Chars: Paper9630 = M8027+A1493+T110
11:27:15 [INFO] Tokens: Paper2406 = M2006+A373+T27
11:27:15 [INFO] Context limit 22912 Available 11646 = CT22912-MT8192-FT3074
11:27:28 [INFO] HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.1 200 OK"
11:27:44 [INFO] HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.1 200 OK"
11:27:44 [INFO] Response received len=1347
11:27:44 [INFO] [llm]   PXD004010_PubText.json -> 58/73 fields filled in row 0 (+13 via LLM)
11:27:44 [INFO] Written 10 rows -> /kaggle/working/outpu

In [21]:
# Preparing to run next iteration of second (LLM) stage (gap filling)

DIR_RUN3 = str(OUTPUT_DIR).replace(f"R{RUN}",f"R{(RUN+2)}")
Path(DIR_RUN3).mkdir(parents=True, exist_ok=True)
RUN+2, DIR_RUN3

(3, '/kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R3')

In [22]:
# Running next iteration of second (LLM) stage (gap filling) 

!python main_fill.py --stage llm --fill-from {DIR_RUN2} --max-tokens {MAX_TOKENS} --context-limit {CONTEXT_LIMIT} --llm-dir "{DIR_RUN3}"  --model {LLM_MODEL} --base-url {api_base} --api-key {api_key} --pattern PXD*.json "{TEST_TEXT_DIR}"

11:35:51 [INFO] Pattern 'PXD*.json' in /kaggle/input/competitions/harmonizing-the-data-of-your-data/Test PubText/Test PubText -> 15 file(s)
11:35:51 [INFO] -- [1/15] PXD004010_PubText.json --
11:35:51 [INFO] [llm] Loading rules CSV from /kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R2/PXD004010_PubText.sdrf.csv
11:35:58 [INFO] Deduplicated: 10 row(s) → 1 unique N/A pattern(s)
11:35:58 [INFO] Filling 24 field(s) for group of 10 row(s)…
11:35:58 [INFO] Chars: Paper9630 = M8027+A1493+T110
11:35:58 [INFO] Tokens: Paper2406 = M2006+A373+T27
11:35:58 [INFO] Context limit 22912 Available 12195 = CT22912-MT8192-FT2525
11:36:11 [INFO] HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.1 200 OK"
11:36:20 [INFO] HTTP Request: POST http://127.0.0.1:8080/v1/chat/completions "HTTP/1.1 200 OK"
11:36:20 [INFO] Response received len=487
11:36:20 [INFO] [llm]   PXD004010_PubText.json -> 59/73 fields filled in row 0 (+1 via LLM)
11:36:20 [INFO] Written 10 rows -> /kaggle/working/output_

### Any improvements over the iterative runs ?

In [23]:
import pandas as pd
from src.eval import score, dataframe_diff, print_column_value_diffs

pxds = pd.read_csv(SAMPLE_SUB)["PXD"].unique()
for pxd in pxds:
    run1 = pd.read_csv(Path(DIR_RUN2) / f"{pxd}_PubText.sdrf.csv")
    run1["PXD"] = pxd
    run2 = pd.read_csv(Path(DIR_RUN3) / f"{pxd}_PubText.sdrf.csv")
    run2["PXD"] = pxd

    f1, harm_norm, harm_subm, eval_df = score(run1, run2, row_id_column_name="ID")
    print(f"## {pxd} F1={f1}")
    display(eval_df.loc[eval_df["f1"]<0.8].head())
    
    diff = dataframe_diff(run1, run2)
    if diff.shape[0] > 0:
        #diff.to_csv(str(SUBMISSION).replace(".csv", "_normalized_diff.csv"), index=False)
        print(f"## {pxd} Column differences")
        print_column_value_diffs(diff)

## PXD004010 F1=0.9861111111111112


,pxd,AnnotationType,precision,recall,f1,jacc
34,PXD004010,Comment[GradientTime],0.0,0.0,0.0,0.0


## PXD004010 Column differences

Column: Comment[GradientTime]
not applicable → 90 min
## PXD050621 F1=0.9305555555555556


,pxd,AnnotationType,precision,recall,f1,jacc
34,PXD050621,Comment[GradientTime],0.0,0.0,0.0,0.0
35,PXD050621,Comment[EnrichmentMethod],0.0,0.0,0.0,0.0
40,PXD050621,Comment[PrecursorMassTolerance],0.0,0.0,0.0,0.0
45,PXD050621,Characteristics[GrowthRate],0.0,0.0,0.0,0.0
62,PXD050621,Characteristics[PooledSample],0.0,0.0,0.0,0.0


## PXD050621 Column differences

Column: Characteristics[GrowthRate]
not applicable → exponential phase

Column: Characteristics[PooledSample]
not applicable → no

Column: Comment[EnrichmentMethod]
not applicable → IMAC

Column: Comment[GradientTime]
not applicable → 90 min

Column: Comment[PrecursorMassTolerance]
not applicable → 10 ppm
## PXD062014 F1=0.8948412698412699


...

,pxd,AnnotationType,precision,recall,f1,jacc
45,PXD025663,Characteristics[GrowthRate],0.0,0.0,0.0,0.0


## PXD025663 Column differences

Column: Characteristics[GrowthRate]
not applicable → exponential phase
## PXD019519 F1=0.9861111111111112


,pxd,AnnotationType,precision,recall,f1,jacc
67,PXD019519,Characteristics[Depletion],0.0,0.0,0.0,0.0


## PXD019519 Column differences

Column: Characteristics[Depletion]
not applicable → NQO1-activatable compound causes an irreversible collapse of the UPS, accompanied by a general accumulation of ubiquitylated proteins
## PXD061285 F1=1.0


,pxd,AnnotationType,precision,recall,f1,jacc


## Combine into submission

In [24]:
RUN=1
SDRF_FILES_OUTPUT = OUTPUT_DIR

RUN=2
SDRF_FILES_OUTPUT=DIR_RUN2

RUN=3
SDRF_FILES_OUTPUT=DIR_RUN3

In [25]:
# --- CONFIG ---
output_folder = Path(SDRF_FILES_OUTPUT)  # folder with all SDRF CSVs
sample_csv = Path(SAMPLE_SUB)  # reference sample CSV
SUBMISSION_MODEL = str(SUBMISSION).replace(".csv", f"_{LLM_MODEL}_T{TEMPERATURE}_R{RUN}.csv")

# --- load reference columns ---
sample_df = pd.read_csv(sample_csv)
cols_order = sample_df.columns.tolist()  # only columns to keep, including PXD and ID

# --- iterate SDRF CSVs ---
all_dfs = []
text_dir = Path(TEST_TEXT_DIR)
files = sorted(text_dir.glob("PXD*.json"))
# we want them same order as in the text dor
for i, jf in enumerate(files, 1):
    pxd = jf.name.split("_")[0]
    # Extract PXD from filename
    f = output_folder / f"{pxd}_PubText.sdrf.csv"
    df = pd.read_csv(f)
    # Extract PXD from filename
    try:
        #pxd = f.stem.split("_")[0]
        df["PXD"] = pxd  # fill the PXD column
        all_dfs.append(df)
    except Exception as err:
        log.error(err)
    
# --- concatenate all files ---
combined_df = pd.concat(all_dfs, ignore_index=True)

# --- ensure all sample columns exist ---
for c in cols_order:
    if c not in combined_df.columns:
        combined_df[c] = pd.NA

# --- assign consecutive IDs ---
if "ID" in cols_order:
    combined_df["ID"] = range(1, len(combined_df) + 1)

# --- keep only columns from sample CSV and in order ---
combined_df = combined_df[cols_order]
# no idea, but this is in SampleSubmission
combined_df['Usage'] = ['Public' if i % 2 == 0 else 'Private' for i in range(len(combined_df))]
combined_df = combined_df.replace('not applicable', 'Not Applicable')

# not all factors are same as characteristics
# it's more complicated to be handled later 
# do not override fv , LLM should handle them in main
#fv_headers = ["Bait","CellPart","Compound","Disease","FractionIdentifier","GeneticModification","Temperature","Treatment"]
#for fv in fv_headers:
#    c = 'Comment' if fv == "FractionIdentifier" else 'Characteristics'
#    combined_df[f'FactorValue[{fv}]'] = combined_df[f'{c}[{fv}]'] 

# defaults, mandatory for proteomics
mandatory_col = "Characteristics[CleavageAgent]"
combined_df.loc[combined_df[mandatory_col] == "Not Applicable", mandatory_col] = "AC=MS:1001251; NT=Trypsin"
mandatory_col = "Comment[AcquisitionMethod]"
combined_df.loc[combined_df[mandatory_col] == "Not Applicable", mandatory_col] = "DDA"

# ---  save final combined CSV ---
combined_df.to_csv(SUBMISSION_MODEL, index=False)
print(f"Combined SDRF dataframe saved -> {SUBMISSION_MODEL} dataset shape {combined_df.shape}")

Combined SDRF dataframe saved -> /kaggle/working/submission_gemma-3-4b-it_T0_R3.csv dataset shape (1659, 81)


## Normalise with keyword maps and Ontology Lookup Service 
using sdrf_pipeline.ols and modelmess cv_map 

In [26]:
from src.cv_map import build_cv_normaliser, normalise_submission
cv_normalizer = build_cv_normaliser(use_ols=True) 
assert "NT=Orbitrap;AC=MS:1000484", cv_normalizer.normalise('Comment[MS2MassAnalyzer]', 'Orbitrap')

assert "Homo sapiens" ==  cv_normalizer.normalise('Characteristics[Organism]', 'human')
assert "Homo sapiens" ==  cv_normalizer.normalise('Characteristics[Organism]', "Homo sapiens")
print(cv_normalizer.normalise('Characteristics[Sex]', "fEmale"))
print(cv_normalizer.normalise('Characteristics[CellLine]', "HeLa"))
print(cv_normalizer.normalise('Characteristics[CellType]', "HeLa"))


2026-03-30 11:43:41,067 INFO No cached ontology files found. Downloading from GitHub...
2026-03-30 11:43:41,067 INFO Downloading ontology file: bto.parquet
2026-03-30 11:43:41,392 INFO Successfully cached: /root/.cache/sdrf-pipelines/ontologies/bto.parquet
2026-03-30 11:43:41,393 INFO Downloading ontology file: chebi.parquet
2026-03-30 11:43:41,750 INFO Successfully cached: /root/.cache/sdrf-pipelines/ontologies/chebi.parquet
2026-03-30 11:43:41,750 INFO Downloading ontology file: cl.parquet
2026-03-30 11:43:42,098 INFO Successfully cached: /root/.cache/sdrf-pipelines/ontologies/cl.parquet
2026-03-30 11:43:42,099 INFO Downloading ontology file: clo.parquet
2026-03-30 11:43:42,538 INFO Successfully cached: /root/.cache/sdrf-pipelines/ontologies/clo.parquet
2026-03-30 11:43:42,538 INFO Downloading ontology file: doid.parquet
2026-03-30 11:43:42,817 INFO Successfully cached: /root/.cache/sdrf-pipelines/ontologies/doid.parquet
2026-03-30 11:43:42,818 INFO Downloading ontology file: efo.par

female
hela
HeLa


#### Make sure this warning disappears - otherwise we are not using the Ontology Lookup Service

WARNING OLS dependencies not available — using keyword maps only

### Normalise

In [27]:
from src.eval import dataframe_diff, score, print_column_value_diffs
submission_df = pd.read_csv(SUBMISSION_MODEL)
SUBMISSION_NORM = str(SUBMISSION_MODEL).replace(".csv", "_normalized.csv")
submission_norm = normalise_submission(submission_df, cv_normalizer)
submission_norm.to_csv(SUBMISSION_NORM, index=False)
submission_norm.head()

2026-03-30 11:43:49,177 INFO DictBackend: loaded bto (6566 terms)
2026-03-30 11:43:49,237 INFO DictBackend: loaded unimod (1561 terms)
2026-03-30 11:43:49,324 INFO DictBackend: loaded ms (4015 terms)


,ID,PXD,Raw Data File,Characteristics[Age],Characteristics[AlkylationReagent],Characteristics[AnatomicSiteTumor],Characteristics[AncestryCategory],Characteristics[BMI],Characteristics[Bait],Characteristics[BiologicalReplicate],...,FactorValue[Bait],FactorValue[CellPart],FactorValue[Compound],FactorValue[ConcentrationOfCompound].1,FactorValue[Disease],FactorValue[FractionIdentifier],FactorValue[GeneticModification],FactorValue[Temperature],FactorValue[Treatment],Usage
0,1,PXD004010,ad_pl03.raw,adult,IAA,breast,European,Not Applicable,FLAG-EGFP,3,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Public
1,2,PXD004010,ad_pl04.raw,adult,IAA,breast,European,Not Applicable,FLAG-EGFP,4,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Private
2,3,PXD004010,ad_pl10.raw,adult,IAA,breast,European,Not Applicable,FLAG-EGFP,10,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Public
3,4,PXD004010,ad_pl07.raw,adult,IAA,breast,European,Not Applicable,FLAG-EGFP,7,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Private
4,5,PXD004010,ad_pl08.raw,adult,IAA,breast,European,Not Applicable,FLAG-EGFP,8,...,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Public


### Verify what the normaliser did

In [28]:
from src.eval import score, dataframe_diff, print_column_value_diffs
f1, harm_norm, harm_subm, eval_df = score(submission_norm, submission_df, row_id_column_name="ID")
print(f"## F1={f1}")
display(eval_df.loc[eval_df["f1"]<0.8].head())

diff = dataframe_diff(submission_df, submission_norm)
#diff.to_csv(str(SUBMISSION).replace(".csv", "_normalized_diff.csv"), index=False)
print("## Column differences")
print_column_value_diffs(diff)

## F1=0.8675925925925926


,pxd,AnnotationType,precision,recall,f1,jacc
3,PXD061285,Comment[IonizationType],0.0,0.0,0.0,0.0
7,PXD061285,Characteristics[AlkylationReagent],0.0,0.0,0.0,0.0
16,PXD061285,Characteristics[CellLine],0.0,0.0,0.0,0.0
17,PXD061285,Characteristics[ReductionReagent],0.0,0.0,0.0,0.0
18,PXD061285,Comment[AcquisitionMethod],0.0,0.0,0.0,0.0


## Column differences

Column: Characteristics[AlkylationReagent]
iodoacetamide → IAA
Iodoacetamide → IAA
Chloroacetamide → CAA

Column: Characteristics[CellLine]
ANBL6 → anbl6
HeLa → hela
U2OS → u2os
MCF-7 → mcf7
HEK293T → hek293t
PC3 → pc3

Column: Characteristics[CleavageAgent]
Trypsin/LysC → AC=MS:1001251;NT=Trypsin|AC=MS:1001309;NT=Lys-C
Trypsin → AC=MS:1001251;NT=Trypsin

Column: Characteristics[Label]
label free sample → AC=MS:1002038;NT=label free sample
TMT16-126 → AC=PRIDE:0000543;NT=TMT16plex
TMT16-127N → AC=PRIDE:0000543;NT=TMT16plex
TMT16-127C → AC=PRIDE:0000543;NT=TMT16plex
TMT16-128N → AC=PRIDE:0000543;NT=TMT16plex
TMT16-128C → AC=PRIDE:0000543;NT=TMT16plex
TMT16-129N → AC=PRIDE:0000543;NT=TMT16plex
TMT16-129C → AC=PRIDE:0000543;NT=TMT16plex
TMT16-130N → AC=PRIDE:0000543;NT=TMT16plex
TMT16-130C → AC=PRIDE:0000543;NT=TMT16plex
TMT16-131N → AC=PRIDE:0000543;NT=TMT16plex
TMT16-131C → AC=PRIDE:0000543;NT=TMT16plex
TMT16-132N → AC=PRIDE:0000543;NT=TMT16plex
TMT16-132C → AC=PR

In [29]:
SUBMISSION_MODEL

'/kaggle/working/submission_gemma-3-4b-it_T0_R3.csv'

In [30]:
ls -al /kaggle/working/

total 13636
drwxr-xr-x  5 root root    4096 Mar 30 11:08 ./
drwxr-xr-x  6 root root    4096 Mar 30 11:08 ../
drwxr-xr-x  5 root root    4096 Mar 30 11:08 modelmess/
drwxr-xr-x 12 root root    4096 Mar 30 11:08 output_sdrfs/
-rw-r--r--  1 root root 1447283 Mar 30 11:08 submission_deepseek-coder-v2-lite-instruct_T0_R3.csv
-rw-r--r--  1 root root 1865622 Mar 30 11:08 submission_deepseek-coder-v2-lite-instruct_T0_R3_normalized.csv
-rw-r--r--  1 root root 1592679 Mar 30 11:08 submission_gemma-3-12b-it_T0_R3.csv
-rw-r--r--  1 root root 1985458 Mar 30 11:08 submission_gemma-3-12b-it_T0_R3_normalized.csv
-rw-r--r--  1 root root 1457787 Mar 30 11:43 submission_gemma-3-4b-it_T0_R3.csv
-rw-r--r--  1 root root 1855433 Mar 30 11:43 submission_gemma-3-4b-it_T0_R3_normalized.csv
-rw-r--r--  1 root root 1681913 Mar 30 11:08 submission_medgemma-4b-it_T0_R3.csv
-rw-r--r--  1 root root 2045152 Mar 30 11:08 submission_medgemma-4b-it_T0_R3_normalized.csv
drwxr-xr-x  2 root root    4096 Mar 30 11:08 .virtua

In [31]:
from src.eval import score, dataframe_diff, print_column_value_diffs

# compare with gpt 5.4 if exists

if Path("/kaggle/working/submission_gpt-5.4_T0_R3_normalized.csv").exists():
    
    df1 = pd.read_csv("/kaggle/working/submission_gpt-5.4_T0_R2_normalized.csv")
    df2 = pd.read_csv("/kaggle/working/submission_gpt-5.4_T0_R3_normalized.csv")
    f1, harm_norm, harm_subm, eval_df = score(df1, df2, row_id_column_name="ID")
    print(f"## F1={f1}")
    display(eval_df.loc[eval_df["f1"]<0.8].head())
    
    diff = dataframe_diff(df1, df2)
    if diff.shape[0]==0:
       print("No differences!") 
    else:
        diff.head()
        #diff.to_csv(str(SUBMISSION).replace(".csv", "_normalized_diff.csv"), index=False)
        print("## Column differences")
        print_column_value_diffs(diff)

# Compare with previous submissions

In [32]:
!ls /kaggle/working 

modelmess
output_sdrfs
submission_deepseek-coder-v2-lite-instruct_T0_R3.csv
submission_deepseek-coder-v2-lite-instruct_T0_R3_normalized.csv
submission_gemma-3-12b-it_T0_R3.csv
submission_gemma-3-12b-it_T0_R3_normalized.csv
submission_gemma-3-4b-it_T0_R3.csv
submission_gemma-3-4b-it_T0_R3_normalized.csv
submission_medgemma-4b-it_T0_R3.csv
submission_medgemma-4b-it_T0_R3_normalized.csv


In [33]:
def reorder(df, df_reference):
    # Create a tuple key temporarily
    df_keys = list(zip(df['PXD'], df['Raw Data File']))
    ref_keys = list(zip(df_reference['PXD'], df_reference['Raw Data File']))

    # map reference order to df indices
    key_to_index = {}
    for i, k in enumerate(df_keys):
        # allow duplicates: store as list of indices
        key_to_index.setdefault(k, []).append(i)

    # build new order
    new_order = []
    used = {}  # keep track of used duplicates
    for k in ref_keys:
        if k in key_to_index:
            idx_list = key_to_index[k]
            used_count = used.get(k, 0)
            if used_count < len(idx_list):
                new_order.append(idx_list[used_count])
                used[k] = used_count + 1

    # reorder df
    return df.iloc[new_order].reset_index(drop=True)

In [34]:
from src.eval import calculate_fill_stability , suggest_next_fills
next = suggest_next_fills(submission_norm)
display(next)

,PXD,Column,Priority,Suggested_Val,Gain_Potential
261,PXD061195,FactorValue[GeneticModification],Medium (Discovery),LLM_NEEDED,1376
89,PXD061195,Characteristics[Modification].6,Medium (Discovery),LLM_NEEDED,1376
48,PXD061195,Characteristics[GrowthRate],Medium (Discovery),LLM_NEEDED,1376
63,PXD061195,Characteristics[Modification].4,Medium (Discovery),LLM_NEEDED,1376
246,PXD061195,FactorValue[FractionIdentifier],Medium (Discovery),LLM_NEEDED,1376
...,...,...,...,...,...
62,PXD061136,Characteristics[Modification].4,Medium (Discovery),LLM_NEEDED,2
243,PXD061009,FactorValue[FractionIdentifier],Medium (Discovery),LLM_NEEDED,2
245,PXD061136,FactorValue[FractionIdentifier],Medium (Discovery),LLM_NEEDED,2
258,PXD061009,FactorValue[GeneticModification],Medium (Discovery),LLM_NEEDED,2


In [35]:
from src.postprocessing import competition_aware_merge, merge_with_similarity
from src.eval import calculate_fill_stability , suggest_next_fills

BEST_SO_FAR = Path("/kaggle/input/datasets/n10705013/best-hdd2026-submissions-rule-first-then-llm")
for f in BEST_SO_FAR.glob("*.csv"):
    df = pd.read_csv(f)[sample_df.columns]
    submission_norm_same_order = reorder(submission_norm, df)
    print(f"=== {f.name} ===")
    f1, harm_norm, harm_subm, eval_df = score(df, submission_norm_same_order, row_id_column_name="ID")
    print(f"=== {f1} {f.name}")
    display(eval_df.loc[eval_df["f1"]<0.8].head())
    diff = dataframe_diff(df, submission_norm_same_order)
    #print_column_value_diffs(diff)    
    #submission_norm_same_order.to_csv("/kaggle/working/debug.csv", index=False)

    #let's merge , fill in the gaps with the best one and see if it improves
    df_merged = competition_aware_merge(df_peak=submission_norm_same_order, df_new=df)
    result = calculate_fill_stability(submission_norm_same_order, df)
    f1_merged, harm_norm, harm_subm, eval_df_merged = score(df, df_merged, row_id_column_name="ID")
    print(f"=== F1={f1} {f.name} --> F1={f1_merged} (merged)")
    print(result)
    next = suggest_next_fills(df_merged)
    display(next)
    #next.to_csv(str(SUBMISSION).replace(".csv","_next.csv"))    

=== localai_submission_gemma-3-4b-it_T0_R3_normalized.csv ===
=== 0.8747300215982722 localai_submission_gemma-3-4b-it_T0_R3_normalized.csv


,pxd,AnnotationType,precision,recall,f1,jacc
3,PXD061285,FactorValue[Disease],0.0,0.0,0.0,0.0
7,PXD061285,FactorValue[Bait],0.0,0.0,0.0,0.0
22,PXD061285,FactorValue[FractionIdentifier],0.0,0.0,0.0,0.0
23,PXD061285,FactorValue[GeneticModification],0.0,0.0,0.0,0.0
30,PXD025663,FactorValue[Disease],0.0,0.0,0.0,0.0


--- Metric Analysis ---
Recall Gains: 13032 holes filled.
Precision Risks: 0 clusters broken.
=== F1=0.8747300215982722 localai_submission_gemma-3-4b-it_T0_R3_normalized.csv --> F1=1.0 (merged)
{'gains': 13032, 'corruptions': 0, 'perfect': 89952}


,PXD,Column,Priority,Suggested_Val,Gain_Potential
15,PXD061195,Characteristics[BMI],Medium (Discovery),LLM_NEEDED,1376
39,PXD061195,Characteristics[DiseaseTreatment],Medium (Discovery),LLM_NEEDED,1376
48,PXD061195,Characteristics[GrowthRate],Medium (Discovery),LLM_NEEDED,1376
32,PXD061195,Characteristics[DevelopmentalStage],Medium (Discovery),LLM_NEEDED,1376
27,PXD061195,Characteristics[Depletion],Medium (Discovery),LLM_NEEDED,1376
...,...,...,...,...,...
150,PXD061136,Comment[CollisionEnergy],Medium (Discovery),LLM_NEEDED,2
158,PXD061136,Comment[FragmentMassTolerance],Medium (Discovery),LLM_NEEDED,2
153,PXD061136,Comment[EnrichmentMethod],Medium (Discovery),LLM_NEEDED,2
172,PXD061136,FactorValue[ConcentrationOfCompound].1,Medium (Discovery),LLM_NEEDED,2


=== submission_bggpt-gemma-3-27b-fp8_gpt-4o-mini123.csv ===
=== 0.53330142882166 submission_bggpt-gemma-3-27b-fp8_gpt-4o-mini123.csv


...

=== submission_2535.csv ===
=== 0.580109188948968 submission_2535.csv


,pxd,AnnotationType,precision,recall,f1,jacc
0,PXD061285,Characteristics[Modification],0.5,0.25,0.333333,0.5
3,PXD061285,FactorValue[Disease],0.0,0.00,0.000000,0.0
4,PXD061285,Characteristics[Staining],0.0,0.00,0.000000,0.0
6,PXD061285,Characteristics[PooledSample],0.0,0.00,0.000000,0.0
7,PXD061285,FactorValue[CellPart],0.0,0.00,0.000000,0.0


/kaggle/working/modelmess/sdrf_pipeline/src/postprocessing.py:233: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result.loc[mask, col] = df_new.loc[mask, col]


--- Metric Analysis ---
Recall Gains: 18350 holes filled.
Precision Risks: 46892 clusters broken.
=== F1=0.580109188948968 submission_2535.csv --> F1=0.7320229596472692 (merged)
{'gains': 18350, 'corruptions': 46892, 'perfect': 43060}


,PXD,Column,Priority,Suggested_Val,Gain_Potential
15,PXD061195,Characteristics[BMI],Medium (Discovery),LLM_NEEDED,1376
35,PXD061195,Characteristics[DiseaseTreatment],Medium (Discovery),LLM_NEEDED,1376
43,PXD061195,Characteristics[GrowthRate],Medium (Discovery),LLM_NEEDED,1376
26,PXD061195,Characteristics[Depletion],Medium (Discovery),LLM_NEEDED,1376
28,PXD061195,Characteristics[DevelopmentalStage],Medium (Discovery),LLM_NEEDED,1376
...,...,...,...,...,...
178,PXD061009,FactorValue[ConcentrationOfCompound].1,Medium (Discovery),LLM_NEEDED,2
198,PXD061009,FactorValue[Temperature],Medium (Discovery),LLM_NEEDED,2
193,PXD061136,FactorValue[GeneticModification],Medium (Discovery),LLM_NEEDED,2
206,PXD061136,FactorValue[Treatment],Medium (Discovery),LLM_NEEDED,2


# Archive

In [36]:
!zip -r {OUTPUT_DIR}.zip {OUTPUT_DIR}

updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/ (stored 0%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD019519_PubText.sdrf.csv (deflated 85%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD050621_PubText.sdrf.json (deflated 96%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD061009_PubText.sdrf.json (deflated 82%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD061090_PubText.sdrf.json (deflated 93%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD061285_PubText.sdrf.csv (deflated 97%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD061009_PubText.sdrf.csv (deflated 74%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD064564_PubText.sdrf.json (deflated 97%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD025663_PubText.sdrf.csv (deflated 91%)
updating: kaggle/working/output_sdrfs/gemma-3-4b-it/T0/R1/PXD061285_PubText.sdrf.json (deflated 98%)
updating: kaggle/working

In [37]:
if server:
    server.stop()

LocalAI server stopped.


In [38]:
end_time = time.monotonic()
print("Execution time [HH:MM:SS.ms]:", timedelta(seconds = end_time - start_time))
print("Execution time [seconds]:", round(timedelta(seconds = end_time - start_time).total_seconds()))

Execution time [HH:MM:SS.ms]: 0:36:17.536168
Execution time [seconds]: 2178
